# best_seg.pt → Hailo-8 HEF (as close to the .pt as possible)

**Run all** (Runtime → Run all) and leave it. It builds a few quantised versions of the model, checks which one scores potholes most like `best_seg.pt`, compiles only that one, and saves **`best_seg_hailo_model.zip`** to Drive. Then copy it to the Pi as before (README step 8).

**Before running:**
1. **Runtime → Change runtime type → T4 GPU.**
2. In Drive folder **`hailo_compile`**: `best_seg.pt`, `calib.zip`, the DFC 3.33 wheel, and `compile_hef.sh`, `eval_quant.py`, `compile_from_har.py` from `Pi C code/hailo/`.
3. It takes about 2–2.5 h. Colab disconnects tabs it thinks are idle, so click into the notebook now and then.

In [ ]:
# 1. Setup
import glob, os, re, shutil, subprocess, time
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
assert gpu, 'No GPU: Runtime -> Change runtime type -> T4 GPU, then Run all again'
print('GPU:', gpu)

from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/hailo_compile'
os.makedirs('/content/build', exist_ok=True)
for f in ['best_seg.pt', 'calib.zip', 'compile_hef.sh', 'eval_quant.py', 'compile_from_har.py']:
    assert os.path.exists(f'{D}/{f}'), f'{f} is missing from Drive/hailo_compile'
    shutil.copy(f'{D}/{f}', '/content/build/')
wheels = sorted(glob.glob(f'{D}/hailo_dataflow_compiler-3.33*.whl'))
assert wheels, 'put the Dataflow Compiler 3.33 wheel in the hailo_compile folder'
WHEEL = shutil.copy(wheels[0], '/content/build/')
assert 'OPTIMIZE_ONLY' in open('/content/build/compile_hef.sh').read(), 'compile_hef.sh in Drive is an old one: copy the new one from Pi C code/hailo/'
%cd /content/build
PY = '/content/build/hef_build/venv/bin/python'

def run(cmd, env=None, log_name='log'):
    """Runs a command, streaming its output (minus TensorFlow noise), saving the log to Drive. Returns the output."""
    t0, text = time.time(), []
    p = subprocess.Popen(cmd, env={**os.environ, **(env or {})}, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        text.append(line)
        if 'layout failed' not in line and 'Unable to register' not in line:
            print(line, end='', flush=True)
    p.wait()
    open(f'{D}/{log_name}.log', 'w').write(''.join(text))
    print(f'---------- {log_name}: exit {p.returncode} after {(time.time() - t0) / 60:.0f} min ----------', flush=True)
    return p.returncode, ''.join(text)

In [ ]:
# 2. Quantise each version (~30 min). 'plain' is the HEF on the Pi now, kept as the baseline.
VARIANTS = [
    ('plain',           {}),
    ('a16cls',          {'A16_CLS': '1'}),
    ('finetune',        {'FINETUNE': '1'}),
    ('finetune_a16cls', {'FINETUNE': '1', 'A16_CLS': '1'}),
]
for name, env in VARIANTS:
    print(f'\n========== quantising {name} ==========', flush=True)
    run(['bash', 'compile_hef.sh', 'best_seg.pt', 'calib.zip', WHEEL], {**env, 'OPTIMIZE_ONLY': '1'}, f'quantise_{name}')
!ls -la /content/build/hef_build/*.har

In [ ]:
# 3. Pick the version closest to best_seg.pt (~15 min), then compile it (~1-1.5 h)
hars = sorted(glob.glob('/content/build/hef_build/*.har'))
assert hars, 'no quantised models: see the errors in step 2'
code, out = run([PY, 'eval_quant.py', '/content/build/hef_build/calib', *hars], None, 'eval_quant')
best = re.search(r'Keeps the borderline 0.4-0.6 band best: (\S+)', out)
assert code == 0 and best, 'the comparison failed: see eval_quant.log in Drive'
best = best.group(1)
har = '/content/build/hef_build/best_seg_hailo_model' + ('' if best == 'plain' else f'_{best}') + '.har'
print(f'\n========== compiling {best} ==========', flush=True)
code, _ = run([PY, 'compile_from_har.py', har], None, f'compile_{best}')
assert code == 0, f'compile failed: see compile_{best}.log in Drive'
shutil.copy(har.replace('.har', '.zip'), f'{D}/best_seg_hailo_model.zip')
print(f'\nDone: Drive/hailo_compile/best_seg_hailo_model.zip (the {best} version). Copy it to the Pi as in README step 8.')